# MentalChat16K — AI Therapeutic Quality Rater

**Purpose:** Use Claude as an AI rater to score a stratified sample of real vs. synthetic mental health counselling responses on therapeutic quality dimensions.

**Rubric based on:** Rogerian core conditions + Motivational Interviewing (MITI) principles

**Dimensions rated (each 1–5):**
- **Empathy** — Does the response acknowledge and reflect the client's emotional state?
- **Clarity** — Is the response easy to understand and free of jargon?
- **Therapeutic Relevance** — Does it address what the client actually said/asked?
- **Autonomy Support** — Does it respect the client's agency rather than being prescriptive?
- **Safety Awareness** — Is it appropriate and non-harmful in a mental health context?

**Sample:** 250 real + 250 synthetic (stratified, random seed for reproducibility)

**Output:** `mc16k_ai_rated_500.csv` — original columns + 5 dimension scores + overall mean

---
**Before running:** Set your `ANTHROPIC_API_KEY` in the cell below (or as an environment variable).

## 0. Install Dependencies

In [ ]:
%pip install anthropic pandas tqdm --quiet

## 1. Configuration

In [ ]:
import os

# ── SET YOUR API KEY HERE ──────────────────────────────────────────────────────
# Option A: paste directly (remove before sharing your notebook!)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

# Option B: already set in your environment — nothing to do here.
# ──────────────────────────────────────────────────────────────────────────────

# Paths
INPUT_CSV   = 'mentalchat16k_full_features.csv'   # your existing feature CSV
OUTPUT_CSV  = 'mc16k_ai_rated_500.csv'            # scored output
CHECKPOINT  = 'mc16k_rating_checkpoint.csv'       # partial saves (resumable)

# Sampling
N_PER_CLASS = 250      # 250 real + 250 synthetic = 500 total
RANDOM_SEED = 42

# Model — Haiku is fast and cheap; good for bulk rating
MODEL       = 'claude-haiku-4-5-20251001'
MAX_TOKENS  = 256      # scores + brief justification fits easily
RATE_LIMIT_DELAY = 0.5 # seconds between calls (adjust if you hit rate limits)

print('Config OK.')
print(f'API key present: {bool(os.environ.get("ANTHROPIC_API_KEY"))}')

## 2. Load & Sample Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(INPUT_CSV, encoding='utf-8')
print(f'Full dataset shape: {df.shape}')
print(f'Source counts:\n{df["source"].value_counts()}')

In [ ]:
# Stratified sample: N_PER_CLASS from each source
real_sample  = df[df['source'] == 'real'].sample(n=N_PER_CLASS, random_state=RANDOM_SEED)
synth_sample = df[df['source'] == 'synthetic'].sample(n=N_PER_CLASS, random_state=RANDOM_SEED)

sample_df = pd.concat([real_sample, synth_sample]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'Sample shape: {sample_df.shape}')
print(f'Source counts in sample:\n{sample_df["source"].value_counts()}')
sample_df[['input', 'output', 'source']].head(3)

## 3. Therapeutic Quality Rubric

The rubric below is the exact prompt sent to Claude for each response.
It is grounded in:
- **Rogers (1957)** — empathy, unconditional positive regard, congruence
- **MITI 4.0** — autonomy support, collaboration, evocation
- **Sharma et al. (2020)** — NLP operationalisation of empathy

In [ ]:
SYSTEM_PROMPT = """You are an expert evaluator of mental health counselling responses.
You assess responses according to evidence-based therapeutic principles drawn from 
Person-Centred Therapy (Rogers, 1957) and Motivational Interviewing (MITI 4.0).
You are precise, consistent, and return only valid JSON."""

RATING_TEMPLATE = """You are evaluating a mental health counsellor's response to a client message.

CLIENT MESSAGE:
{client_input}

COUNSELLOR RESPONSE:
{counsellor_output}

Rate the COUNSELLOR RESPONSE on the following five dimensions.
Use a scale of 1 (very poor) to 5 (excellent) for each.

DIMENSIONS:
1. empathy          — Does the response acknowledge and reflect the client's emotional state with warmth and understanding?
2. clarity          — Is the response clear, accessible, and free of unhelpful jargon?
3. therapeutic_relevance — Does the response directly address what the client expressed or asked about?
4. autonomy_support — Does the response respect the client's agency and avoid being prescriptive or directive?
5. safety_awareness — Is the response appropriate and non-harmful for a mental health context? (penalise dismissiveness, false reassurance, or unsafe advice)

Respond ONLY with a valid JSON object in exactly this format — no preamble, no markdown:
{{
  "empathy": <1-5>,
  "clarity": <1-5>,
  "therapeutic_relevance": <1-5>,
  "autonomy_support": <1-5>,
  "safety_awareness": <1-5>,
  "justification": "<one sentence explaining the most important strength or weakness>"
}}"""

print('Rubric defined.')

## 4. Rating Function

In [ ]:
import anthropic
import json
import time
import re

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

SCORE_KEYS = ['empathy', 'clarity', 'therapeutic_relevance', 'autonomy_support', 'safety_awareness']

def rate_response(client_input: str, counsellor_output: str) -> dict:
    """
    Send one (input, output) pair to Claude for therapeutic quality rating.
    Returns a dict with 5 scores + justification, or error flags on failure.
    """
    prompt = RATING_TEMPLATE.format(
        client_input=str(client_input).strip()[:1500],    # truncate very long inputs
        counsellor_output=str(counsellor_output).strip()[:2000]
    )

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = response.content[0].text.strip()

        # Strip markdown code fences if present
        raw = re.sub(r'^```json\s*|^```\s*|```$', '', raw, flags=re.MULTILINE).strip()

        parsed = json.loads(raw)

        # Validate all expected keys exist and scores are in range
        scores = {}
        for key in SCORE_KEYS:
            val = parsed.get(key)
            if not isinstance(val, (int, float)) or not (1 <= val <= 5):
                raise ValueError(f'Invalid score for {key}: {val}')
            scores[key] = float(val)

        scores['justification'] = str(parsed.get('justification', '')).strip()
        scores['rating_error']  = False
        return scores

    except Exception as e:
        # Return NaN scores so the row is still saved and identifiable
        return {
            **{key: float('nan') for key in SCORE_KEYS},
            'justification': f'ERROR: {str(e)[:100]}',
            'rating_error': True
        }


# Quick smoke test on one row
test_row = sample_df.iloc[0]
test_result = rate_response(test_row['input'], test_row['output'])
print('Smoke test result:')
print(json.dumps(test_result, indent=2))

## 5. Rate Full Sample (Resumable)

This cell saves a checkpoint after every 10 rows, so you can safely interrupt and re-run without losing progress.

In [ ]:
from tqdm.notebook import tqdm

# ── Resume logic ──────────────────────────────────────────────────────────────
# If a checkpoint exists, load it and skip already-rated rows.
if os.path.exists(CHECKPOINT):
    rated_df = pd.read_csv(CHECKPOINT)
    already_done = set(rated_df.index.tolist()) if 'original_index' not in rated_df.columns \
                   else set(rated_df['original_index'].tolist())
    print(f'Resuming from checkpoint: {len(rated_df)} rows already rated.')
else:
    rated_df = pd.DataFrame()
    already_done = set()
    print('Starting fresh.')

# Tag sample_df with its position for resume tracking
sample_df['original_index'] = sample_df.index

results = []
checkpoint_interval = 10

to_rate = sample_df[~sample_df['original_index'].isin(already_done)]
print(f'Rows to rate: {len(to_rate)}')

for i, (_, row) in enumerate(tqdm(to_rate.iterrows(), total=len(to_rate), desc='Rating')):
    scores = rate_response(row['input'], row['output'])

    result_row = row.to_dict()
    result_row.update(scores)
    results.append(result_row)

    time.sleep(RATE_LIMIT_DELAY)

    # Save checkpoint every N rows
    if (i + 1) % checkpoint_interval == 0:
        checkpoint_so_far = pd.concat(
            [rated_df, pd.DataFrame(results)], ignore_index=True
        )
        checkpoint_so_far.to_csv(CHECKPOINT, index=False)

# Final save
final_df = pd.concat([rated_df, pd.DataFrame(results)], ignore_index=True)

# Compute overall mean therapeutic quality score
final_df['therapeutic_quality_mean'] = final_df[SCORE_KEYS].mean(axis=1)

final_df.to_csv(OUTPUT_CSV, index=False)

# Clean up checkpoint
if os.path.exists(CHECKPOINT):
    os.remove(CHECKPOINT)

print(f'\nDone. Rated {len(final_df)} rows.')
print(f'Errors: {final_df["rating_error"].sum()}')
print(f'Saved to: {OUTPUT_CSV}')
final_df[['source'] + SCORE_KEYS + ['therapeutic_quality_mean']].head(5)

## 6. Summary Statistics — Real vs Synthetic

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Load output if re-running from saved file
if 'final_df' not in dir():
    final_df = pd.read_csv(OUTPUT_CSV)

ALL_SCORE_COLS = SCORE_KEYS + ['therapeutic_quality_mean']

summary = final_df.groupby('source')[ALL_SCORE_COLS].agg(['mean', 'std']).round(3)
print('=== Mean (SD) by Source ===')
print(summary.to_string())

## 7. Statistical Tests — Welch's t-test + Cohen's d

In [ ]:
from scipy.stats import ttest_ind

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.var(a, ddof=1) +
                          (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std != 0 else 0

real_scores  = final_df[final_df['source'] == 'real']
synth_scores = final_df[final_df['source'] == 'synthetic']

stat_results = []
for col in ALL_SCORE_COLS:
    r = real_scores[col].dropna()
    s = synth_scores[col].dropna()
    stat, p = ttest_ind(r, s, equal_var=False)
    d = cohens_d(r.values, s.values)
    stat_results.append({
        'Dimension': col,
        'Mean (Real)': round(r.mean(), 3),
        'Mean (Synthetic)': round(s.mean(), 3),
        'Diff (Real−Synth)': round(r.mean() - s.mean(), 3),
        't-stat': round(stat, 3),
        'p-value': round(p, 4),
        'Significant': 'Yes' if p < 0.05 else 'No',
        "Cohen's d": round(d, 3),
        'Effect Size': 'large' if abs(d) >= 0.8 else ('medium' if abs(d) >= 0.5 else 'small')
    })

stats_df = pd.DataFrame(stat_results)
print('=== Welch t-test Results ===')
stats_df

## 8. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

palette = {'real': '#4C72B0', 'synthetic': '#DD8452'}

# 8a. Boxplots for all 5 dimensions
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, col in zip(axes, SCORE_KEYS):
    sns.boxplot(
        x='source', y=col, data=final_df,
        palette=palette, ax=ax,
        flierprops=dict(marker='o', markersize=3, alpha=0.4)
    )
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.set_ylim(0.5, 5.5)
    ax.set_xlabel('')
    ax.set_ylabel('Score (1–5)' if ax == axes[0] else '')

plt.suptitle('AI-Rated Therapeutic Quality: Real vs Synthetic', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('therapeutic_quality_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: therapeutic_quality_boxplots.png')

In [ ]:
# 8b. Overall mean therapeutic quality distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    data=final_df, x='therapeutic_quality_mean', hue='source',
    kde=True, bins=20, alpha=0.5, palette=palette, ax=ax
)
ax.set_title('Overall Therapeutic Quality Score Distribution', fontsize=13)
ax.set_xlabel('Mean Score (1–5)')
plt.tight_layout()
plt.savefig('therapeutic_quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: therapeutic_quality_distribution.png')

In [ ]:
# 8c. Radar chart — mean scores by source
import matplotlib.patches as mpatches
from matplotlib.path import Path

labels = [c.replace('_', ' ').title() for c in SCORE_KEYS]
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

real_means  = real_scores[SCORE_KEYS].mean().tolist()  + [real_scores[SCORE_KEYS].mean().tolist()[0]]
synth_means = synth_scores[SCORE_KEYS].mean().tolist() + [synth_scores[SCORE_KEYS].mean().tolist()[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, real_means,  color='#4C72B0', linewidth=2, label='Real')
ax.fill(angles, real_means,  color='#4C72B0', alpha=0.25)
ax.plot(angles, synth_means, color='#DD8452', linewidth=2, label='Synthetic')
ax.fill(angles, synth_means, color='#DD8452', alpha=0.25)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(1, 5)
ax.set_yticks([2, 3, 4, 5])
ax.set_title('Mean Therapeutic Quality: Real vs Synthetic', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))

plt.tight_layout()
plt.savefig('therapeutic_quality_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: therapeutic_quality_radar.png')

## 9. Correlation — AI Ratings vs Linguistic Features

This is the key analysis connecting your existing RQ1/RQ2 work to therapeutic quality.

In [ ]:
LINGUISTIC_FEATURES = [
    'flesch_reading_ease', 'gunning_fog', 'smog', 'dale_chall',
    'polarity', 'subjectivity',
    'negation_count',
    'word_count', 'avg_words_per_sentence', 'avg_chars_per_word',
    'type_token_ratio',
    'hedge_count', 'hedge_ratio',
    'first_person_count', 'second_person_count', 'third_person_count',
    'question_mark_count', 'ellipsis_count'
]

# Keep only columns that actually exist in our rated dataframe
available_ling = [f for f in LINGUISTIC_FEATURES if f in final_df.columns]

corr_matrix = final_df[available_ling + ALL_SCORE_COLS].corr(method='pearson')

# Show correlations between linguistic features and quality scores only
corr_subset = corr_matrix.loc[available_ling, ALL_SCORE_COLS].round(3)
print('=== Pearson Correlations: Linguistic Features vs AI Quality Scores ===')
corr_subset

In [ ]:
# Heatmap of correlations
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    corr_subset,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Linguistic Features vs Therapeutic Quality Scores\n(Pearson r)', fontsize=13)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correlation_heatmap.png')

## 10. Top Linguistic Predictors of Therapeutic Quality

In [ ]:
# Which linguistic features best predict overall therapeutic quality mean?
top_predictors = (
    corr_matrix.loc[available_ling, 'therapeutic_quality_mean']
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
top_predictors.columns = ['Linguistic Feature', '|Pearson r| with Quality Mean']
top_predictors['Pearson r'] = corr_matrix.loc[
    top_predictors['Linguistic Feature'].values, 'therapeutic_quality_mean'
].values.round(3)

print('=== Top Linguistic Predictors of Therapeutic Quality ===')
top_predictors

In [ ]:
# Bar chart of top predictors
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#4C72B0' if r >= 0 else '#DD8452' for r in top_predictors['Pearson r']]
ax.barh(top_predictors['Linguistic Feature'], top_predictors['Pearson r'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r  (positive = higher feature → higher quality)')
ax.set_title('Linguistic Predictors of Overall Therapeutic Quality\n(Blue = positive, Orange = negative association)', fontsize=12)
plt.tight_layout()
plt.savefig('linguistic_predictors_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: linguistic_predictors_quality.png')

## 11. Summary Output

Print a clean summary of key findings for your write-up.

In [ ]:
print('=' * 60)
print('SUMMARY: AI THERAPEUTIC QUALITY RATING')
print('=' * 60)

print(f'\nSample: {len(final_df)} responses '
      f'({len(real_scores)} real, {len(synth_scores)} synthetic)')
print(f'Rating errors: {int(final_df["rating_error"].sum())} '
      f'({100*final_df["rating_error"].mean():.1f}%)')

print('\n--- Mean Therapeutic Quality Scores ---')
for col in ALL_SCORE_COLS:
    r_mean = real_scores[col].mean()
    s_mean = synth_scores[col].mean()
    diff   = r_mean - s_mean
    direction = 'Real HIGHER' if diff > 0 else 'Synthetic HIGHER'
    print(f'  {col:<35} Real={r_mean:.2f}  Synth={s_mean:.2f}  Δ={diff:+.2f}  ({direction})')

print('\n--- Significant Differences (p < 0.05) ---')
sig = stats_df[stats_df['Significant'] == 'Yes']
if len(sig) == 0:
    print('  None')
else:
    for _, row in sig.iterrows():
        print(f"  {row['Dimension']}: p={row['p-value']:.4f}, d={row[\"Cohen's d\"]:.3f} ({row['Effect Size']})")

print('\n--- Top 3 Linguistic Predictors of Quality ---')
for _, row in top_predictors.head(3).iterrows():
    print(f"  {row['Linguistic Feature']}: r = {row['Pearson r']:.3f}")

print(f'\nOutput saved to: {OUTPUT_CSV}')
print('=' * 60)

---
## Notes for Write-Up

**Cite the rubric as:**
> Dimensions operationalised from Rogers (1957) core conditions (empathy, unconditional positive regard), MITI 4.0 (autonomy support), and Sharma et al. (2020) computational empathy framework.

**Limitations to acknowledge:**
- AI rater may share stylistic biases with the AI-generated (synthetic) responses — mention this as a potential confound
- Scores are ordinal (1–5), treated as interval for correlation — standard in psychometric research but note the assumption
- No inter-rater reliability yet — this will be addressed by the peer survey

**Next step:** Correlate peer survey scores with these AI scores to establish convergent validity.